# 🔑 INSTRUCTOR ANSWER KEY — do not distribute

# 🐧 Microproject 4: Palmer Penguins
*Elements of Data Science — Regression Microprojects* <br>

From 2007 to 2009, Dr. Kristen Gorman and the Palmer Station Long Term Ecological Research program measured hundreds of penguins on three islands in the Antarctic Peninsula. Ecologists use easy-to-measure body dimensions to estimate harder-to-measure quantities like body mass.

> **Your research question:** Can a penguin's flipper length predict its body mass — and is the relationship the same for every species?

In Lab 8 you built a **regression toolbox** step by step. This microproject gives you much less scaffolding: you decide how to explore, you build the line, and you judge whether it is any good. Your final product is **one publication-quality figure**, submitted to the class gallery, that your instructor will show in class.

**Review:** [Ch. 15.2 The Regression Line](https://inferentialthinking.com/chapters/15/2/Regression_Line.html), [Ch. 15.1 Correlation](https://inferentialthinking.com/chapters/15/1/Correlation.html)

|Part|What you do|
|---|---|
|1|Rebuild your regression toolbox|
|2|Meet the data|
|3|Look first: scatter plot and a guess|
|4|Fit the line and interpret it|
|5|Check the fit: errors and `np.polyfit`|
|6|The twist: dig deeper|
|7|Make and submit your final figure|

## ⚙️ Initialization — DO NOT EDIT
👉 **Run the cell below first** (Shift+Enter) before anything else.

In [ ]:
# 🔧 Run this cell first! (Shift+Enter)
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "shared-public")))
sys.path.insert(0, "/home/jovyan/shared-public")
from init import *
from microproject_tools import my_project, submit_plot

### 🧰 Standard imports

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt

Enter your name (first and last), then check that this is the project you were assigned.

In [ ]:
name = 'Test Student'
my_project(name)

## 1. Your Regression Toolbox
Copy your four functions from **Lab 8, Questions 1.5 and 4.4** into the cell below (or rebuild them from [Ch. 15.2](https://inferentialthinking.com/chapters/15/2/Regression_Line.html)). Then run the test cell: a perfect line $y = 2x + 1$ must give $r = 1$, slope 2, intercept 1.

In [ ]:
def standard_units(any_numbers):
    "Convert an array of numbers to standard units."
    return (any_numbers - np.mean(any_numbers)) / np.std(any_numbers)

def correlation(t, label_x, label_y):
    "The correlation coefficient r between two columns of Table t."
    return np.mean(standard_units(t.column(label_x)) * standard_units(t.column(label_y)))

def slope(t, label_x, label_y):
    "Slope of the regression line, in original units."
    r = correlation(t, label_x, label_y)
    return r * np.std(t.column(label_y)) / np.std(t.column(label_x))

def intercept(t, label_x, label_y):
    "Intercept of the regression line, in original units."
    return np.mean(t.column(label_y)) - slope(t, label_x, label_y) * np.mean(t.column(label_x))

In [ ]:
# Toolbox test — don't change this cell
test = Table().with_columns('x', make_array(1, 2, 3, 4), 'y', make_array(3, 5, 7, 9))
print('r =', correlation(test, 'x', 'y'), '  slope =', slope(test, 'x', 'y'), '  intercept =', intercept(test, 'x', 'y'))
print('✅ Toolbox works!' if np.isclose(correlation(test, 'x', 'y'), 1) and np.isclose(slope(test, 'x', 'y'), 2)
      and np.isclose(intercept(test, 'x', 'y'), 1) else '❌ Check your functions against Lab 8.')

## 2. Meet the Data
One row per penguin (342 penguins with both measurements):
- **species**: Adelie, Chinstrap, or Gentoo
- **island**: Biscoe, Dream, or Torgersen
- **flipper length (mm)**
- **body mass (g)**
- **sex**

*Source:* Gorman, Williams & Fraser (2014), *PLoS ONE* 9(3): e90081; the [palmerpenguins](https://allisonhorst.github.io/palmerpenguins/) data set (Horst, Hill & Gorman, 2020), CC0.

In [ ]:
data = Table.read_table('penguins.csv')
data

<font color=blue>**Explore.**</font> How many penguins of each species are there? Which species is heaviest on average? Use any Table methods you know (`num_rows`, `sort`, `where`, `group`, `hist`...). Add more cells if you like.

In [ ]:
print(data.num_rows, 'rows')
print('x range:', min(data.column('flipper length (mm)')), 'to', max(data.column('flipper length (mm)')))
print('y range:', min(data.column('body mass (g)')), 'to', max(data.column('body mass (g)')))

## 3. Look First
<font color=blue>**Question 3.1.**</font> Make a scatter plot with **`flipper length (mm)`** on the horizontal axis and **`body mass (g)`** on the vertical axis. No line yet.

In [ ]:
data.scatter('flipper length (mm)', 'body mass (g)')
plt.show()

<font color=green>Describe the **direction**, **form**, and **strength** of the association. Then guess the value of *r*. Do you see any clusters in the cloud?</font>

---

ANSWER

---

## 4. Fit the Line
<font color=blue>**Question 4.1.**</font> Use your toolbox to compute `my_r`, `my_slope`, and `my_intercept` for predicting **body mass (g)** from **flipper length (mm)**.

In [ ]:
my_r = correlation(data, 'flipper length (mm)', 'body mass (g)')
my_slope = slope(data, 'flipper length (mm)', 'body mass (g)')
my_intercept = intercept(data, 'flipper length (mm)', 'body mass (g)')
print(f'r = {my_r:.3f}')
print(f'body mass (g) = {my_slope:.4f} × flipper length (mm) + {my_intercept:.2f}')

<font color=green>Interpret your slope **in words, with units**: "For each additional ___ mm of flipper, the body mass (g) is predicted to change by ___ grams." How close was your guess for *r*?</font>

---

ANSWER

---

<font color=blue>**Question 4.2.**</font> Predict the body mass of a penguin with a **200 mm** flipper. Store your prediction as `my_prediction`.

In [ ]:
my_prediction = my_slope * 200 + my_intercept
my_prediction

## 5. Check the Fit
A line can have a large *r* and still be the wrong model. Plot the **errors** (residuals), `actual y − predicted y`, against $x$. A good straight-line fit leaves errors that look like a patternless cloud centered on 0 ([Ch. 15.5](https://inferentialthinking.com/chapters/15/5/Visual_Diagnostics.html)).

<font color=blue>**Question 5.1.**</font> Compute the array `errors` and make a scatter plot of the errors against $x$, with a horizontal line at 0 (`plt.axhline(0)`).

<details><summary><b>💡 Hint</b></summary>

`errors = data.column(y) - (my_slope * data.column(x) + my_intercept)` — then `plt.scatter(...)` with the x column and `errors`. This is the `line_errors` idea from Lab 8, Question 2.3.

</details>

In [ ]:
x = data.column('flipper length (mm)')
y = data.column('body mass (g)')
errors = y - (my_slope * x + my_intercept)
plt.scatter(x, errors, s=12)
plt.axhline(0, color='red')
plt.xlabel('flipper length (mm)'); plt.ylabel('error')
plt.show()

<font color=green>Do the errors look like a patternless cloud, or do you see a pattern (a curve, a fan, clusters, a few extreme points)? What does that tell you about your straight-line model?</font>

---

ANSWER

---

<font color=blue>**Question 5.2.**</font> Check your slope and intercept with `np.polyfit` (Lab 8, Part 6). Store `True` or `False` in `agrees`.

In [ ]:
coefficients = np.polyfit(data.column('flipper length (mm)'), data.column('body mass (g)'), 1)
agrees = np.isclose(coefficients.item(0), my_slope) and np.isclose(coefficients.item(1), my_intercept)
agrees

## 6. The Twist: One Line or Three?
<font color=blue>**Question 6.1.**</font> Fit a separate regression line for **each species**. Build a Table `by_species` with columns `species`, `r`, `slope`, and `intercept`, one row per species.

<details><summary><b>💡 Hint</b></summary>

Loop over `make_array('Adelie', 'Chinstrap', 'Gentoo')`, use `data.where('species', sp)`, and collect results with `np.append` — or build three rows with `Table().with_rows`.

</details>

In [ ]:
species_names = make_array('Adelie', 'Chinstrap', 'Gentoo')
rs = make_array(); slopes = make_array(); intercepts = make_array()
for sp in species_names:
    t = data.where('species', sp)
    rs = np.append(rs, correlation(t, 'flipper length (mm)', 'body mass (g)'))
    slopes = np.append(slopes, slope(t, 'flipper length (mm)', 'body mass (g)'))
    intercepts = np.append(intercepts, intercept(t, 'flipper length (mm)', 'body mass (g)'))
by_species = Table().with_columns('species', species_names, 'r', rs, 'slope', slopes, 'intercept', intercepts)
by_species

<font color=green>1. Every single species has a **smaller** *r* than all penguins together. How is that possible? (Hint: what does the gap *between* species add to the overall trend?)
2. A biologist finds a new penguin with a 200 mm flipper. Would you use the overall line or a species line to predict its mass? What would you need to know?
3. Do the species lines have similar slopes? What would that mean biologically?</font>

---

ANSWER

---

## 7. Your Final Figure
This is the product of your microproject. Your instructor will show the class gallery on screen, so make a figure a scientist would be proud of. **Requirements:**

- [ ] Scatter plot of the data
- [ ] Your regression line, labeled in the legend with its equation and *r*
- [ ] Axis labels **with units** (e.g. `flipper length (mm)` and `body mass (g)`)
- [ ] A title that states your **finding**, not just the variable names
- [ ] **Also** color each species differently and draw each species' regression line.

Build everything after `fig = plt.figure(...)` so `submit_plot` can save it.

<details><summary><b>💡 Starter pattern</b></summary>

```python
fig = plt.figure(figsize=(9, 5.5))
plt.scatter(x_values, y_values, s=14, alpha=0.6, label='data')
line_x = make_array(low, high)
plt.plot(line_x, my_slope * line_x + my_intercept, color='black', lw=2,
         label=f'y = {my_slope:.3f}x + {my_intercept:.1f},  r = {my_r:.2f}')
plt.xlabel('...'); plt.ylabel('...'); plt.title('...'); plt.legend()
plt.show()
```

</details>

In [ ]:
fig = plt.figure(figsize=(9, 5.5))
colors = make_array('darkorange', 'purple', 'teal')
for i in np.arange(3):
    sp = by_species.column('species').item(i)
    t = data.where('species', sp)
    xs = make_array(min(t.column('flipper length (mm)')), max(t.column('flipper length (mm)')))
    plt.scatter(t.column('flipper length (mm)'), t.column('body mass (g)'), s=14, alpha=0.5, color=colors.item(i))
    plt.plot(xs, by_species.column('slope').item(i) * xs + by_species.column('intercept').item(i), color=colors.item(i), lw=3,
             label=f"{sp}: r = {by_species.column('r').item(i):.2f}")
line_x = make_array(170, 232)
plt.plot(line_x, my_slope * line_x + my_intercept, color='black', ls='--', lw=2,
         label=f'all penguins: y = {my_slope:.1f}x {my_intercept:+.0f}, r = {my_r:.2f}')
plt.xlabel('flipper length (mm)'); plt.ylabel('body mass (g)')
plt.title('Flipper length predicts mass, but species explains much of the trend')
plt.legend(); plt.show()

<font color=blue>**Question 7.1.**</font> Write a **one-sentence caption** stating your main finding, then submit. You may submit again as often as you like — the newest version replaces the old one.

In [ ]:
caption = 'Across all penguins r = 0.87, but within each species r is only 0.47–0.70: much of the overall trend comes from differences between species.'
submit_plot(fig, name, 'MP4', r=my_r, slope=my_slope, intercept=my_intercept, caption=caption)

## 8. Reflection
<font color=green>In 3–5 sentences: What did the data say about your research question? How confident are you, and why? What was hardest to do **without** step-by-step instructions? Be ready to explain your figure to the class in 60 seconds.</font>

---

ANSWER

---

### 🏁 Done!
Save your notebook (💾). Your figure is already in the class gallery. Download and upload an HTML copy to Canvas if your instructor asks for one ([instructions](https://laserchemist.github.io/data/submission-instructions.html#html)).